# 1. Limpieza del corpus

Este notebook es el **primer paso del pipeline**. Parte de los JSON crudos scrapeados
y produce:
- `data/{medio}_limpio.json` — un JSON limpio por cada medio
- `data/corpus_limpio.parquet` — el corpus completo en un único archivo

## 0. Configuración y dependencias

In [1]:
import json
import re
import warnings
import html
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
warnings.filterwarnings('ignore')

In [2]:
if Path.cwd().name == 'notebooks':
    os.chdir('..')

DATA_DIR   = Path('data')
RAW_DIR    = Path('data/raw')
PROC_DIR   = Path('data/processed')
EMB_DIR    = Path('embeddings')
OUTPUT_DIR = Path('output')
for d in [PROC_DIR, EMB_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


MEDIOS = [
    'elconfidencial',
    'eldiario',
    'publico',
    'okdiario',
    'elpais',
    'elmundo',
]

print(f'Working dir: {Path.cwd()}')
print(f'Medios: {MEDIOS}')

Working dir: /Users/blancagallardo/PycharmProjects/TFM
Medios: ['elconfidencial', 'eldiario', 'publico', 'okdiario', 'elpais', 'elmundo']


## 1. Función de limpieza de texto

`limpiar(texto)` normaliza cada campo de texto aplicando los siguientes pasos en orden:

| Paso | Qué hace |
|------|----------|
| `html.unescape` | Convierte entidades HTML (`&amp;` → `&`, `&nbsp;` → ` `, etc.) |
| Caracteres de escape | Elimina `\n`, `\t`, `\r` y comillas escapadas |
| Tags HTML | Elimina cualquier etiqueta `<...>` residual del scraping |
| URLs | Elimina enlaces `http...` que no aportan contenido |
| Caracteres de control | Elimina bytes de control ASCII (0x00–0x1f, 0x7f–0x9f) |
| Artefactos de corte | Elimina palabras truncadas al final de línea (`_`) |
| Puntos suspensivos | Normaliza `...` y variantes a espacio |
| Espacios múltiples | Colapsa cualquier secuencia de espacios a uno solo |


In [3]:
def limpiar(texto):
    if not isinstance(texto, str):
        return ''
    texto = html.unescape(texto)
    texto = texto.replace('\\n', ' ').replace('\\t', ' ').replace('\\"', '"').replace('\\r', ' ')
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'http\S+', '', texto)
    texto = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', texto)
    texto = re.sub(r'\s\w{1,3}[_]$', '', texto)
    texto = re.sub(r'\.\s?\.\s?\.', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [4]:
def cargar_y_limpiar(medio):
    path = RAW_DIR / f'{medio}.json'
    noticias = json.loads(path.read_text(encoding='utf-8'))

    for i, n in enumerate(noticias):
        n['id']       = f'{medio}_{i:05d}'
        n['titular']  = limpiar(n.get('titular', ''))
        n['abstract'] = limpiar(n.get('abstract', ''))
        n['noticia']  = limpiar(n.get('noticia', ''))
        n['fecha']    = n.get('fecha', '')

        autor_raw = n.get('autor', '')
        n['autores_lista'] = (
            [a.strip() for a in autor_raw.split(',') if a.strip()]
            if isinstance(autor_raw, str) else []
        )

    out = PROC_DIR / f'{medio}_limpio.json'
    out.write_text(json.dumps(noticias, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'[{medio}] {len(noticias)} noticias → {out.name}')
    return noticias

noticias_por_medio = {m: cargar_y_limpiar(m) for m in MEDIOS}

todas = [n for v in noticias_por_medio.values() for n in v]
df = pd.DataFrame(todas)
df['fecha_dt'] = pd.to_datetime(df['fecha'], errors='coerce')

# Normalización de nombres de medio (conserva nombres de visualización)
mapping = {
    'El Confidencial': 'El Confidencial',
    'elDiario.es':     'elDiario.es',
    'Público':         'Público',
    'El Mundo':        'El Mundo',
    'El País':         'El País',
    'OKDiario':        'OKDiario',
}
df['medio'] = df['medio'].map(mapping)
print(df['medio'].unique())

df["texto"] = (
    df["titular"].fillna("") + ". " +
    df["abstract"].fillna("") + ". " +
    df["noticia"].fillna("")
).apply(limpiar)

df["len_texto"] = df["texto"].str.len()

print(f'\nTotal artículos: {len(df)}')
print(df.groupby('medio').size().to_string())
print(f"\nLongitud media : {df['len_texto'].mean():.0f} caracteres")
print(f"Longitud máxima: {df['len_texto'].max()} | mínima: {df['len_texto'].min()}")

[elconfidencial] 37178 noticias → elconfidencial_limpio.json
[eldiario] 38637 noticias → eldiario_limpio.json
[publico] 26602 noticias → publico_limpio.json
[okdiario] 55985 noticias → okdiario_limpio.json
[elpais] 68664 noticias → elpais_limpio.json
[elmundo] 31458 noticias → elmundo_limpio.json
<ArrowStringArray>
['El Confidencial', 'elDiario.es', 'Público', 'OKDiario', 'El País',
 'El Mundo']
Length: 6, dtype: str

Total artículos: 258524
medio
El Confidencial    37178
El Mundo           31458
El País            68664
OKDiario           55985
Público            26602
elDiario.es        38637

Longitud media : 4455 caracteres
Longitud máxima: 296086 | mínima: 4


`cargar_y_limpiar(medio)` realiza por cada medio:
1. Lee el JSON crudo de `data/{medio}.json`
2. Asigna un `id` único (`{medio}_{índice:05d}`)
3. Aplica `limpiar()` a los campos `titular`, `abstract` y `noticia`
4. Normaliza el campo `autor` a una lista de autores (`autores_lista`)
5. Guarda el resultado en `data/{medio}_limpio.json`

A continuación se construye el DataFrame global con todos los medios,
se parsea la fecha, se normaliza el nombre del medio y se genera
el campo `texto` (titular + abstract + noticia concatenados).

## 2. Exportación

Se guarda el DataFrame completo como **Parquet** para que el resto de notebooks puedan cargarlo directamente sin repetir la limpieza.

In [6]:
# Guardar corpus combinado
df.to_parquet(PROC_DIR / "corpus_limpio.parquet", index=False)
print(f"Guardado: {PROC_DIR / 'corpus_limpio.parquet'}")
print(f"Shape: {df.shape} | Columnas: {list(df.columns)}")

Guardado: data/corpus_limpio.parquet
Shape: (258524, 14) | Columnas: ['medio', 'url', 'titular', 'autor', 'fecha', 'abstract', 'noticia', 'tema', 'id', 'autores_lista', 'truncado', 'fecha_dt', 'texto', 'len_texto']
